# Sports action phase recognition

**Recipe 91 · Level 3 (Intermediate) · Choice · Multiframe & temporal understanding**

Sources:

- S02: [Decision primitives](https://docs.typesafe.ai/primitives)
- S03: [Confidence](https://docs.typesafe.ai/confidence)
- S09: [Images](https://docs.typesafe.ai/images)
- S11: [Models](https://docs.typesafe.ai/models)
- S12: [Jev 1.14 release](https://typesafe.ai/blog/jev-1-14)

## What you will build

Build a simulated clip annotator for **one stationary basketball set-shot attempt**. Four chronological stills support one bounded `Choice`: what phase is visible **at the last timestamp**? A window may contain several phases; its final phase is the target. Earlier frames distinguish a chest hold from a return to rest after release.

| End phase | Visible definition |
| --- | --- |
| preparation | Ball held near the chest before lifting or release. |
| execution | Ball and hands lifted above the head, or ball just released while hands stay overhead. |
| recovery | Hands lowering or lowered after an earlier visible lift/release; ball still separated. |
| unrecognized | Other movement, hidden final evidence, absent ball, or order incompatible with one attempt. |

These are original **synthetic schematic drawings**, with synthetic answers authored separately. They are neither athlete footage nor measurements of sporting technique. The task makes no biomechanics, coaching, injury, or medical claim. Ordered still images are the documented image input; Python supplies their order and timestamps ([S09](https://docs.typesafe.ai/images)).

## Setup and run mode

A [run mode](../../docs/glossary.md#run-mode) identifies where answers came from. [Provenance](../../docs/glossary.md#provenance) records that origin: all committed answers here are synthetic. Their metrics check this pipeline and do not measure Jev quality. We pin `jev-1.14.0` for optional recording and enable experimental image input ([S11](https://docs.typesafe.ai/models), [S12](https://typesafe.ai/blog/jev-1-14)). No long-context header is needed for four images.

There are 12 validation windows, 12 held-out test windows, and two unscored demonstrations. Each contains seven authored source frames, from which Python chooses four. One call answers one window; 26 windows means 26 logical requests in a complete replay or recording. The local budget is 26 and the live backend also counts retries. The README provides a recording path; interactive live mode below does not report held-out results.

In [ ]:
from IPython.display import display

from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import (
    accuracy,
    confusion_matrix,
    evaluate_outcomes,
    select_confidence_threshold,
)
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path, select_split
from jev_cookbook.simulation import ActionLog, ReviewQueue
from jev_cookbook.style import apply_style, plot_confusion_matrix, run_header, show_answer
from jev_cookbook.vision import run_record

helpers = load_helpers()
examples = load_inputs()
labels = load_labels()
scored = [e for e in examples if e.split in {"validation", "test"}]
backend = get_backend(fixtures=responses_path(), images=True, model="jev-1.14.0")
runner = helpers.WindowRunner(backend, max_windows=len(examples))
results = {}
apply_style()
run_header(91, "Sports action phase recognition", backend=backend, n_examples=len(scored))
check = " — pipeline check, not a Jev result" if backend.mode in {"synthetic", "scripted"} else ""
selection = " — validation selection step"

## The state

Python verifies finite, increasing timestamps before selecting four evenly spaced **frame indices**, including the first and last. With these uniformly spaced source frames, indices 0, 2, 4, 6 land at 0.0, 0.4, 0.8, 1.2 seconds. Index spacing is not a claim about uniform time spacing for irregular footage; a separate limit rejects sampled gaps over 1.5 seconds. Windows exceeding four seconds or source lists outside 4–60 frames are rejected before any call. A real video adapter would decode and timestamp frames upstream.

Only image references, timestamps, and the fixed movement/target description reach inference. The renderer's pose codes, case identifiers, split, and answers remain outside the request. A [replay key](../../docs/glossary.md#replay-key) is the content identity used to retrieve a stored response; it includes the actual supplied image bytes. Paths use opaque clip names. The display below shows exactly the selected pixels, with timestamps added outside the input images.

In [ ]:
demos = select_split(examples, "demo")
demo = demos[0]
state, questions, assets = helpers.build_request(demo.fields)
print({k: v for k, v in state.items() if k != "frames"})
print("Selected timestamps:", [f["timestamp_seconds"] for f in state["frames"]])
print("Frame references supplied:", len(state["frames"]))
display(helpers.contact_sheet(demo.fields))

## The questions

One `Choice` carries the four fixed alternatives ([S02](https://docs.typesafe.ai/primitives)). It judges one end phase, rather than independently tagging each frame and asking Python to guess which phase the user meant. The allowed single-attempt progression is preparation → execution → recovery. Skipping a sampled phase is allowed; returning to holding the ball after release falls outside this window definition.

`unrecognized` covers both another movement and insufficient evidence. It defers the annotation to [review](../../docs/glossary.md#review), a request for a person to decide, at any confidence. A misleading sign inside a frame remains scene content, never a new instruction.

In [ ]:
print(questions["end_phase"].instructions)
for option, description in questions["end_phase"].criteria.items():
    print(f"{option}: {description}")

## One answer up close

Read the typed answer and its probability distribution before looking at aggregate results. The first demonstration ends after release with hands down. The second contains waving arms and no ball. The stored distributions were written for this recipe; the confidence field is derived by the shared typed-answer code, following the documented Choice calculation ([S03](https://docs.typesafe.ai/confidence)). Confidence is not a guarantee of correctness.

In [ ]:
for example in demos:
    results[example.id] = runner.decide(example.fields)
    print(example.id)
    show_answer(results[example.id]["end_phase"])
display(helpers.contact_sheet(demos[1].fields))

## Python's part

A [gold label](../../docs/glossary.md#gold-label) is the authored expected answer used for checking. Its reason is stored beside it in `labels.jsonl`. A [confidence gate](../../docs/glossary.md#confidence-gate) accepts a recognized phase only when its confidence reaches a cutoff. `unrecognized`, malformed confidence, and unsupported choices always defer. The threshold cannot turn them into annotations.

Select the cutoff on validation only. The candidate family is the distinct observed confidence values for recognized validation answers. The objective is **maximum answered fraction subject to 100% correct accepted validation answers** (`target_accuracy=1.0`); ties prefer the lower cutoff. This is a small synthetic selection exercise, not a promise that future accepted answers are all correct. Freeze the selected number before opening the test results. If no recognized answer exists or no observed cutoff meets the objective, freeze a disabled policy (`None`) that sends every window to review. This is an explicit policy state, not a fabricated confidence value.

[Coverage](../../docs/glossary.md#coverage) is the fraction of windows accepted; [accuracy](../../docs/glossary.md#accuracy) is the fraction correct among the stated denominator. We use actual policy outcomes with `evaluate_outcomes`, since a confidence-only comparison would incorrectly count high-confidence `unrecognized` answers. All annotation entries are simulated in-memory `ActionLog` records; all deferrals enter `ReviewQueue`. No footage or external annotation service is changed.

In [ ]:
validation = select_split(examples, "validation")
for example in validation:
    results[example.id] = runner.decide(example.fields)
named = [e for e in validation if results[e.id]["end_phase"].choice in helpers.PHASES]
threshold = None
if named:
    try:
        threshold = select_confidence_threshold(
            [results[e.id]["end_phase"].choice == labels[e.id]["end_phase"] for e in named],
            [results[e.id]["end_phase"].confidence for e in named],
            target_accuracy=1.0,
        )
    except ValueError:
        threshold = None
frozen_threshold = threshold
cutoff_text = "disabled: all review" if threshold is None else f"{threshold:.4f}"
validation_outcomes = [
    helpers.classify(e.id, results[e.id]["end_phase"], frozen_threshold) for e in validation
]
validation_metrics = evaluate_outcomes(
    [o.accepted for o in validation_outcomes],
    [results[e.id]["end_phase"].choice == labels[e.id]["end_phase"] for e in validation],
)
print(
    f"Frozen cutoff={cutoff_text}, candidate pool N={len(named)}, validation N={len(validation)}{selection}{check}"
)
print(
    f"Accepted={validation_metrics.n_answered}/{validation_metrics.n_total}, coverage={validation_metrics.coverage:.4f}, accepted event accuracy={validation_metrics.accuracy:.4f}{selection}{check}"
)
actions, reviews = ActionLog(), ReviewQueue()
for example in demos:
    helpers.annotate(
        example.id, results[example.id]["end_phase"], frozen_threshold, actions, reviews
    )
print(
    "Simulated annotation:",
    [{k: row[k] for k in ("kind", "payload", "executed")} for row in actions.to_dicts()],
)
print("Review:", [{"item": row["item"], "reason": row["reason"]} for row in reviews.to_dicts()])

## Evaluation

**Event accuracy** here means one correct end-phase answer per window, not per frame. We report raw four-way event accuracy over all test windows and accepted event accuracy over the policy's answered windows; abstention is the fraction sent to review. [Risk](../../docs/glossary.md#risk) is one minus accepted accuracy. A correct `unrecognized` is correct in the raw four-way task, but is still an abstention for simulated annotation.

The deterministic baseline reads the **same four images**. It finds orange ball and pink hand pixels, compares their centroids, and uses hand lowering over time to distinguish recovery. This is a meaningful restricted pose heuristic, with no answer or renderer coordinates as inputs. Its exact colors, fixed pixel thresholds, and inability to validate a single-attempt chronology limit it to these drawings. The baseline defers `unrecognized` without a confidence score. All baseline constants were fixed before test evaluation.

Test is opened once below after the policy is frozen. Reusing these cached results for the matrix and failure inspection makes no further calls. Different scene offsets, scale, and backgrounds ensure no decoded frame is duplicated across validation and test. Both splits still share the same drawing grammar, so they are not independent real-world sporting events.

In [ ]:
test = select_split(examples, "test")
test_outcomes = []
test_actions, test_reviews = ActionLog(), ReviewQueue()
if backend.mode == "live":
    print(
        "Unrecorded live exploration: held-out evaluation is not reported. Use the README recorder path."
    )
else:
    for example in test:
        results[example.id] = runner.decide(example.fields)
        test_outcomes.append(
            helpers.annotate(
                example.id,
                results[example.id]["end_phase"],
                frozen_threshold,
                test_actions,
                test_reviews,
            )
        )
    gold = [labels[e.id]["end_phase"] for e in test]
    predicted = [results[e.id]["end_phase"].choice for e in test]
    baseline = [helpers.pixel_pose_baseline(e.fields) for e in test]
    metrics = evaluate_outcomes(
        [o.accepted for o in test_outcomes], [a == b for a, b in zip(gold, predicted, strict=True)]
    )
    baseline_metrics = evaluate_outcomes(
        [p in helpers.PHASES for p in baseline],
        [a == b for a, b in zip(gold, baseline, strict=True)],
    )
    for name, guesses, summary in (
        ("Choice policy", predicted, metrics),
        ("Pixel pose baseline", baseline, baseline_metrics),
    ):
        print(
            f"{name}, test N={len(test)}: raw event accuracy={accuracy(gold, guesses):.4f}{check}"
        )
        print(
            f"{name}, test N={summary.n_total}: accepted={summary.n_answered}, abstentions={summary.n_total - summary.n_answered}, abstention fraction={1 - summary.coverage:.4f}, coverage={summary.coverage:.4f}{check}"
        )
        print(
            f"{name}, accepted N={summary.n_answered}: event accuracy={summary.accuracy:.4f}, risk={summary.risk:.4f}{check}"
        )
    print(
        f"Simulated annotations={len(test_actions)}, review entries={len(test_reviews)}, test N={len(test)}{check}"
    )

In [ ]:
if backend.mode != "live":
    matrix = confusion_matrix(gold, predicted, labels=helpers.OPTIONS)
    print(f"Raw end-phase counts, test N={len(test)}; rows=gold, columns=predicted{check}")
    print("Order:", matrix.labels)
    print(matrix.matrix.tolist())
    fig = plot_confusion_matrix(
        matrix.matrix, matrix.labels, title=f"End phase, test N={len(test)}{check}"
    )
    display(fig)

### A confidence gate still accepts a wrong event

A deliberately confident stored answer mistakes a new chest hold for the preparation stage of the original attempt. The earlier released ball makes that order incompatible with this window's single-attempt definition. Inspect the images, raw answer, gold reason, and actual outcome together. This is the failure a single last-frame pose classifier cannot resolve. Other errors include choosing execution for a mixed window that ends in recovery and reacting to misleading scene text; their lower confidence sends them to review.

In [ ]:
if backend.mode != "live":
    wrong = [
        (e, o)
        for e, o in zip(test, test_outcomes, strict=True)
        if o.accepted and o.label != labels[e.id]["end_phase"]
    ]
    print(f"Accepted wrong events={len(wrong)}, test N={len(test)}{check}")
    for example, outcome in wrong:
        print(example.id, "gold:", labels[example.id], "accepted:", outcome.accepted)
        show_answer(results[example.id]["end_phase"])
        display(helpers.contact_sheet(example.fields))

## What was and was not measured

The cell below reports mode, model and capture dates when available, and N for validation, held-out test, and demonstrations. Synthetic measurements are pipeline checks only. **Not measured live.** We made no model-quality, inference-latency, token-use, or cost measurement. Unknown accounting fields remain `None`; local replay duration is never called inference latency.

Optional recorded runs retain the model and UTC date actually returned by the API. They describe only those recorded examples with the frozen policy. Interactive live exploration is not a reportable held-out benchmark; its test branch above is suppressed. Record first and replay the captured responses to report that sample. Camera cuts, ball tracking through long occlusions, multiple athletes, and technique assessment are outside this lesson.

In [ ]:
accounting = [run_record(result, mode=backend.mode) for result in results.values()]
dates = sorted(
    {
        result["end_phase"].provenance.date
        for result in results.values()
        if result["end_phase"].provenance.date
    }
)
print("Mode:", backend.mode)
print("Returned model:", backend.model if backend.mode in {"recorded", "live"} else None)
print("Capture dates:", dates or None)
print(f"N validation={len(validation)}, N held-out test={len(test_outcomes)}, N demo={len(demos)}")
print(
    f"Logical requests={runner.attempts}, selected frame appearances={runner.attempts * helpers.SAMPLE_COUNT}"
)
print("Accounting example:", accounting[0])
if backend.mode in {"synthetic", "scripted"}:
    print("Not measured live: a pipeline check, not a Jev result.")
    print(f"N: validation={len(validation)}, held-out test={len(test_outcomes)}, demo={len(demos)}")
else:
    print(
        "Recorded metadata names the returned model and actual capture dates; unknown measurements stay None."
    )
    print(f"N: validation={len(validation)}, held-out test={len(test_outcomes)}, demo={len(demos)}")

## Next steps

Collect consented clips with separate movement attempts, annotate final phases without seeing model answers, and reserve a new test set before changing the phase definitions or sampling schedule. Camera movement and unknown actions deserve separate error counts on that dataset. A richer baseline could estimate pose and track the ball while keeping the same end-of-window target.

[Sentiment classification](../01-sentiment-classification/) introduces a bounded Choice and threshold selection. The local [vision helpers](../../docs/vision.md) explain image identity and ordered still frames, and the [live guide](../../docs/live.md) explains capture metadata.